# Colab 02 (Full): Step LoRA Training (Inline)
- `unified_trainer.py` JSSP step 학습 워크플로우를 노트북형으로 전체화
- HF/local dataset source 지원
- resume_from_checkpoint / shuffle_data / output_dir 자동생성 / 업로드 포함


In [ ]:
!pip -q install -U unsloth
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets trl peft accelerate bitsandbytes wandb


## 1) 설정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import csv
import random
from datetime import datetime
from functools import partial
from pathlib import Path
import math
import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder
from unsloth import FastLanguageModel

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 8192,
    # 'model_type': 'llama8b',
    'model_type': 'llama8b',
    # 'model_type': 'qwen25_7b',
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # LoRA
    'lora_r': 128,
    'lora_alpha': 128,
    'lora_dropout': 0.0,
    'bias': 'none',
    'use_gradient_checkpointing': 'unsloth',
    'random_state': 42,
    'use_rslora': True,
    'loftq_config': None,

    # train hparams
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 2,
    'per_device_eval_batch_size': 8,
    'num_train_epochs': 2,
    'learning_rate': 5e-5,
    'save_steps': 1000,
    'save_strategy': 'steps',
    'save_total_limit': 20,
    'logging_steps': 10,
    'eval_steps': 100,
    'warmup_steps': 20,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 42,
    'group_by_length': False,
    'action_loss_weight': 4.0,
    # 'dataset_num_proc': 64,
    'dataset_num_proc': 16,

    # speed controls
    'enable_eval': False,
    'eval_split_mode': 'fixed_per_size',
    'eval_instances_per_size': 3,
    'eval_split_ratio': 0.05,
    # 'max_train_samples': None,
    'max_train_samples': 100000,
    'min_train_feasible_actions': 2,
    'min_eval_feasible_actions': 1,
    'max_eval_samples': 20,
    'eval_subset_seed': 42,

    # task flags (unified_trainer.py 호환)
    'train_jssp': True,
    'train_fssp': False,
    'train_vrp_tsp': False,
    'train_knapsack': False,
    'train_binpack': False,
    'train_lm_head': False,
    'train_embed_tokens': False,


    # policy head
    'policy_head_type': 'candidate_scoring',  # token_generation | candidate_scoring
    'candidate_score_token': '<CAND_SCORE>',
    'candidate_score_head_init_std': 0.02,
    'candidate_scoring_do_sample': True,
    'candidate_scoring_temperature': 1.0,
    'candidate_scoring_query_forward_batch_size': 16,

    # action-token row initialization
    'reinit_action_token_rows_when_frozen': True,
    'action_token_reinit_scale': 0.02,
    'action_token_reinit_seed': 42,
    'action_token_reinit_share_input_output_rows': True,

    # training-time action probability probe
    'log_action_prob_during_train': True,
    'action_prob_probe_steps': 20,
    'action_prob_probe_rows': 5,
    'action_prob_probe_topk': 8,

    # environment mode
    'env_mode': 'dispatch',  # serial | dispatch

    # adapter role
    'adapter_role': 'policy',  # policy | reason | mixed
    'action_code_width': 4,
    'action_code_cap': 9999,
    'step_supervision_mode': 'action_only',  # resolved automatically from adapter_role

    # unified_trainer.py 옵션 동등화
    'step_dataset_path': None,
    'fp16': None,
    'bf16': None,
    'dataloader_num_workers': 0,
    'evaluation_strategy': 'steps',
    'load_best_model_at_end': False,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'remove_unused_columns': False,
    'report_to': 'wandb',
    'run_name': None,

    # dataset source
    'dataset_source': 'hf',
    'step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'policy_step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'policy_step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'policy_step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'reason_step_dataset_hf_repo': 'HYUNJINI/jssp_reason_step_train_all_v1',
    'reason_step_dataset_hf_file': 'train_data/jssp_step_train_reason.jsonl',
    'reason_step_dataset_local_path': '/content/jssp_step_train_reason.jsonl',
    'mixed_step_dataset_hf_repo': 'HYUNJINI/jssp_mixed_step_train_all_v1',
    'mixed_step_dataset_hf_file': 'train_data/jssp_step_train_with_reason.jsonl',
    'mixed_step_dataset_local_path': '/content/jssp_step_train_with_reason.jsonl',
    # 'policy_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
    'policy_step_dataset_hf_repo_dispatch': 'HYUNJINI/AXXXX_jssp_policy_step_train_dispatch_v1',
    'policy_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_policy_dispatch.jsonl',
    'policy_step_dataset_local_path_dispatch': '/content/jssp_step_train_policy_dispatch.jsonl',
    'reason_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_reason_step_train_dispatch_v1',
    'reason_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_reason_dispatch.jsonl',
    'reason_step_dataset_local_path_dispatch': '/content/jssp_step_train_reason_dispatch.jsonl',
    'mixed_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_mixed_step_train_dispatch_v1',
    'mixed_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_with_reason_dispatch.jsonl',
    'mixed_step_dataset_local_path_dispatch': '/content/jssp_step_train_with_reason_dispatch.jsonl',

    # data options
    # 'shuffle_data': False,
    'shuffle_data': True,
    'shuffle_seed': 42,

    # output
    'output_accord': False,
    'output_list_of_lists': False,
    # 'output_dir': None,
    'output_dir': '/content/drive/MyDrive/LLM_JSSP_checkpoints/fix_jssp_policy_dispatch_llama8b_r128',
    'auto_unique_output_dir': True,
    'output_dir_suffix': None,

    # resume
    'resume_from_checkpoint': None,
    # 'resume_from_checkpoint': '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128/checkpoint-19000',

    # wandb
    'enable_wandb': False,
    'wandb_project': None,

    # optional upload
    'upload_after_train': False,
    'upload_on_save': True,
    'hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'policy_hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason_hf_model_repo_out': 'HYUNJINI/jssp_reason_llama8b_step_r64_ep2',
    'upload_source': 'latest_checkpoint',  # final | latest_checkpoint | checkpoint_tag | output_dir
    'checkpoint_tag': 'checkpoint-200',
}

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.1-8B-Instruct',
    'qwen25_7b': 'unsloth/Qwen3.5-9B-Base',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
    'qwen25_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
}

print(CFG)


from transformers import TrainerCallback


def resolve_task_type(cfg):
    if cfg.get('train_jssp', False):
        return 'jssp'
    if cfg.get('train_fssp', False):
        return 'fssp'
    if cfg.get('train_vrp_tsp', False):
        return 'vrp_tsp'
    if cfg.get('train_knapsack', False):
        return 'knapsack'
    if cfg.get('train_binpack', False):
        return 'binpack'
    return 'generic'


def resolve_output_dir(cfg, task_type):
    cached = cfg.get('_resolved_output_dir')
    if cached:
        return os.path.expanduser(str(cached))

    explicit = cfg.get('output_dir')
    if explicit:
        base_dir = Path(os.path.expanduser(str(explicit)))
    else:
        env_mode = str(cfg.get('env_mode', 'serial')).lower()
        adapter_role = str(cfg.get('adapter_role', 'policy')).lower()
        model_type = str(cfg.get('model_type', 'model')).lower()
        lora_r = int(cfg.get('lora_r', 0))
        run_name = cfg.get('run_name')
        if run_name:
            folder = str(run_name)
        else:
            folder = f"{task_type}_{adapter_role}_{env_mode}_{model_type}_r{lora_r}"
        base_dir = Path('outputs') / folder

    resolved = base_dir
    if not cfg.get('resume_from_checkpoint') and bool(cfg.get('auto_unique_output_dir', False)):
        suffix = cfg.get('output_dir_suffix')
        if suffix in (None, ''):
            suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
        resolved = base_dir.parent / f"{base_dir.name}_{suffix}"

    cfg['_resolved_output_dir'] = str(resolved)
    return str(resolved)


def resolve_upload_dir(output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
    output_dir = Path(output_dir)
    if upload_source == 'output_dir':
        return output_dir
    if upload_source == 'final':
        final_dir = output_dir / 'final'
        return final_dir if final_dir.exists() else output_dir
    checkpoints = sorted(
        [p for p in output_dir.glob('checkpoint-*') if p.is_dir()],
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
    )
    if upload_source == 'checkpoint_tag':
        if checkpoint_tag:
            tagged = output_dir / str(checkpoint_tag)
            if tagged.exists():
                return tagged
            raise FileNotFoundError(f"checkpoint_tag not found: {tagged}")
        raise ValueError('checkpoint_tag upload_source requires CFG[\'checkpoint_tag\']')
    if checkpoints:
        return checkpoints[-1]
    final_dir = output_dir / 'final'
    if final_dir.exists():
        return final_dir
    return output_dir


def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    trainable_percent = (trainable_model_params / all_model_params * 100) if all_model_params > 0 else 0.0
    print(f"총 학습가능 매개변수: {trainable_model_params:,}개")
    print(f"총 매개변수: {all_model_params:,}개")
    print(f"학습가능 매개변수 비율: {trainable_percent:.2f}%")
    return {
        'trainable_model_params': int(trainable_model_params),
        'all_model_params': int(all_model_params),
        'trainable_percent': float(trainable_percent),
    }


class UploadCheckpointToHFCallback(TrainerCallback):
    def __init__(self, repo_id, token, output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
        self.repo_id = repo_id
        self.token = token
        self.output_dir = Path(output_dir)
        self.upload_source = upload_source
        self.checkpoint_tag = checkpoint_tag
        self.api = HfApi(token=token) if token else None

    def on_save(self, args, state, control, **kwargs):
        if not self.token:
            print('UploadCheckpointToHFCallback: HF_TOKEN empty, skip upload')
            return control
        try:
            self.api.create_repo(repo_id=self.repo_id, repo_type='model', private=False, exist_ok=True)
            folder = resolve_upload_dir(
                self.output_dir,
                upload_source=self.upload_source,
                checkpoint_tag=self.checkpoint_tag or getattr(args, 'checkpoint_tag', None),
            )
            if not folder.exists():
                print(f'UploadCheckpointToHFCallback: upload folder missing -> {folder}')
                return control
            upload_folder(
                repo_id=self.repo_id,
                repo_type='model',
                folder_path=str(folder),
                token=self.token,
                commit_message=f"Auto-upload checkpoint at step {state.global_step}",
            )
            print(f'UploadCheckpointToHFCallback: uploaded {folder} -> {self.repo_id}')
        except Exception as exc:
            print(f'UploadCheckpointToHFCallback warning: {exc}')
        return control


## 2) 유틸 함수 (inline)


In [ ]:
import re
from typing import Any, Dict, List, Sequence

# --- Inline helpers aligned with source action-centric step supervision ---

# --- Action special-token helpers (standalone notebook) ---
ACTION_TOKEN_PREFIX = "A"
LEGACY_ACTION_TOKEN_PREFIX = "S"
ACTION_CODE_PATTERN = re.compile(r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>", re.IGNORECASE)


def format_action_code(code_index: int, code_width: int = 4, prefix: str = ACTION_TOKEN_PREFIX):
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"


def parse_action_code(text: str, code_width: int = 4):
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)


def build_action_special_tokens(code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    return [
        format_action_code(code_idx, code_width=code_width, prefix=prefix)
        for code_idx in range(int(code_start), int(code_cap) + 1)
    ]


def ensure_action_special_tokens(tokenizer, model=None, code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    action_tokens = build_action_special_tokens(
        code_width=code_width,
        code_start=code_start,
        code_cap=code_cap,
        prefix=prefix,
    )
    existing = set(getattr(tokenizer, "additional_special_tokens", []) or [])
    missing = [tok for tok in action_tokens if tok not in existing]
    added = 0
    if missing:
        added = int(tokenizer.add_special_tokens({"additional_special_tokens": missing}))

    if model is not None:
        target_size = len(tokenizer)
        input_size = int(model.get_input_embeddings().weight.shape[0])
        if input_size != target_size:
            model.resize_token_embeddings(target_size)

    return {
        "num_added_tokens": int(added),
        "vocab_size": int(len(tokenizer)),
        "action_token_count": int(len(action_tokens)),
    }


def maybe_reinitialize_action_token_rows(
    tokenizer,
    model,
    train_lm_head: bool,
    train_embed_tokens: bool,
    code_width: int = 4,
    code_start: int = 1,
    code_cap: int = 9999,
    prefix: str = ACTION_TOKEN_PREFIX,
    enabled: bool = True,
    scale: float = 0.02,
    seed: int | None = None,
    share_input_output_rows: bool = True,
):
    if not bool(enabled):
        return {
            "applied": False,
            "reason": "disabled",
        }
    if bool(train_lm_head) or bool(train_embed_tokens):
        return {
            "applied": False,
            "reason": "trainable_rows",
        }

    start_token = format_action_code(code_start, code_width=code_width, prefix=prefix)
    end_token = format_action_code(code_cap, code_width=code_width, prefix=prefix)
    start_id = action_code_to_token_id(tokenizer, start_token)
    end_id = action_code_to_token_id(tokenizer, end_token)
    if int(end_id) < int(start_id):
        raise ValueError(
            f"Invalid action token range: start_id={start_id}, end_id={end_id}"
        )

    input_embeddings = model.get_input_embeddings()
    output_embeddings = model.get_output_embeddings()
    if input_embeddings is None or getattr(input_embeddings, 'weight', None) is None:
        raise RuntimeError('Input embedding weights are unavailable for action-token reinitialization.')
    if output_embeddings is None or getattr(output_embeddings, 'weight', None) is None:
        raise RuntimeError('Output embedding weights are unavailable for action-token reinitialization.')

    input_weight = input_embeddings.weight
    output_weight = output_embeddings.weight
    if int(start_id) <= 0:
        raise ValueError('Action token ids unexpectedly start at 0; cannot build reference distribution.')

    row_count = int(end_id) - int(start_id) + 1
    if row_count <= 0:
        raise ValueError(f'Invalid action row count: {row_count}')

    device_list = []
    if input_weight.is_cuda:
        device_list.append(input_weight.device)

    with torch.random.fork_rng(devices=device_list):
        if seed is not None:
            torch.manual_seed(int(seed))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed))

        with torch.no_grad():
            ref_input = input_weight[: int(start_id)].detach().float()
            input_mean = ref_input.mean(dim=0, keepdim=True)
            input_std = ref_input.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-6)
            input_noise = torch.randn(
                (row_count, input_weight.shape[1]),
                device=input_weight.device,
                dtype=torch.float32,
            )
            new_input = input_mean + (input_std * input_noise * float(scale))
            input_weight[int(start_id): int(end_id) + 1] = new_input.to(dtype=input_weight.dtype)

            if bool(share_input_output_rows):
                if int(output_weight.shape[1]) != int(new_input.shape[1]):
                    raise RuntimeError(
                        'Cannot share action input/output rows because dimensions differ: '
                        f'input_dim={int(new_input.shape[1])}, output_dim={int(output_weight.shape[1])}'
                    )
                new_output = new_input.to(device=output_weight.device, dtype=torch.float32)
            else:
                ref_output = output_weight[: int(start_id)].detach().float()
                output_mean = ref_output.mean(dim=0, keepdim=True)
                output_std = ref_output.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-6)
                output_noise = torch.randn(
                    (row_count, output_weight.shape[1]),
                    device=output_weight.device,
                    dtype=torch.float32,
                )
                new_output = output_mean + (output_std * output_noise * float(scale))
            output_weight[int(start_id): int(end_id) + 1] = new_output.to(dtype=output_weight.dtype)

    return {
        "applied": True,
        "reason": "frozen_action_rows_reinitialized",
        "start_id": int(start_id),
        "end_id": int(end_id),
        "row_count": int(row_count),
        "scale": float(scale),
        "seed": None if seed is None else int(seed),
        "share_input_output_rows": bool(share_input_output_rows),
    }


def summarize_action_token_row_geometry(
    tokenizer,
    model,
    sample_tokens=None,
    code_width: int = 4,
):
    if sample_tokens is None:
        sample_tokens = [
            format_action_code(1, code_width=code_width),
            format_action_code(2, code_width=code_width),
            format_action_code(3, code_width=code_width),
            format_action_code(100, code_width=code_width),
            format_action_code(1000, code_width=code_width),
            format_action_code(9999, code_width=code_width),
        ]

    action_ids = [int(action_code_to_token_id(tokenizer, tok)) for tok in sample_tokens]
    input_weight = model.get_input_embeddings().weight.detach().float().cpu()
    output_weight = model.get_output_embeddings().weight.detach().float().cpu()

    input_pairwise = []
    output_pairwise = []
    for i in range(len(action_ids)):
        for j in range(i + 1, len(action_ids)):
            input_pairwise.append(float(torch.norm(input_weight[action_ids[i]] - input_weight[action_ids[j]], p=2).item()))
            output_pairwise.append(float(torch.norm(output_weight[action_ids[i]] - output_weight[action_ids[j]], p=2).item()))

    input_norms = [float(torch.norm(input_weight[token_id], p=2).item()) for token_id in action_ids]
    output_norms = [float(torch.norm(output_weight[token_id], p=2).item()) for token_id in action_ids]

    return {
        'sample_tokens': list(sample_tokens),
        'action_ids': list(action_ids),
        'input_pairwise_min': min(input_pairwise) if input_pairwise else 0.0,
        'input_pairwise_max': max(input_pairwise) if input_pairwise else 0.0,
        'input_pairwise_mean': (sum(input_pairwise) / len(input_pairwise)) if input_pairwise else 0.0,
        'output_pairwise_min': min(output_pairwise) if output_pairwise else 0.0,
        'output_pairwise_max': max(output_pairwise) if output_pairwise else 0.0,
        'output_pairwise_mean': (sum(output_pairwise) / len(output_pairwise)) if output_pairwise else 0.0,
        'input_norm_min': min(input_norms) if input_norms else 0.0,
        'input_norm_max': max(input_norms) if input_norms else 0.0,
        'output_norm_min': min(output_norms) if output_norms else 0.0,
        'output_norm_max': max(output_norms) if output_norms else 0.0,
    }


def action_code_to_token_id(tokenizer, action_code: str) -> int:
    token_id = tokenizer.convert_tokens_to_ids(str(action_code))
    if token_id is None:
        raise ValueError(f"Tokenizer returned None token id for action_code={action_code!r}")
    token_id = int(token_id)
    unk_id = getattr(tokenizer, "unk_token_id", None)
    if unk_id is not None and token_id == int(unk_id):
        encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
        if len(encoded) == 1:
            return int(encoded[0])
        raise ValueError(
            f"Action code {action_code!r} is not a single tokenizer token. "
            "Special tokens were likely not installed correctly."
        )
    return token_id


def token_id_to_action_code(tokenizer, token_id: int, code_width: int = 4):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if isinstance(token, bytes):
        token = token.decode("utf-8", errors="ignore")
    token_text = str(token) if token is not None else ""
    parsed = parse_action_code(token_text, code_width=code_width)
    if parsed is not None:
        return parsed
    decoded = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    return parse_action_code(decoded, code_width=code_width)


def validate_action_tokenizer_installation(tokenizer, code_width: int = 4, code_start: int = 1, code_cap: int = 9999):
    for action_code in (
        format_action_code(code_start, code_width=code_width),
        format_action_code(min(code_start + 7, code_cap), code_width=code_width),
        format_action_code(code_cap, code_width=code_width),
    ):
        token_id = action_code_to_token_id(tokenizer, action_code)
        roundtrip = token_id_to_action_code(tokenizer, token_id, code_width=code_width)
        if roundtrip != action_code:
            raise ValueError(
                f"Action token round-trip failed: action_code={action_code}, "
                f"token_id={token_id}, roundtrip={roundtrip}"
            )

ACTION_CODE_EXTRACT_RE = re.compile(r"<\s*A\s*(\d+)\s*>", re.IGNORECASE)


def _normalize_action_code(text: str, code_width: int = 4):
    match = ACTION_CODE_EXTRACT_RE.search(str(text))
    if not match:
        return None
    return f"<A{int(match.group(1)):0{int(code_width)}d}>"


def build_step_messages(example, step_supervision_mode="action_only"):
    state_text = str(example.get("state_text", ""))
    reason_input_text = str(example.get("reason_input_text", ""))
    target_text = str(example.get("target_text", ""))
    reason_target_text = str(example.get("reason_target_text", "")).strip()

    if step_supervision_mode not in {"action_only", "action_reason", "reason_only"}:
        raise ValueError(
            f"Unsupported step_supervision_mode={step_supervision_mode}. "
            "Use one of: action_only, action_reason, reason_only."
        )

    if step_supervision_mode == "reason_only":
        if not reason_input_text:
            raise ValueError("Missing 'reason_input_text' for reason dataset sample.")
        if not reason_target_text:
            raise ValueError("Missing 'reason_target_text' for reason dataset sample.")
        user_content = reason_input_text
        assistant_content = reason_target_text
        system_content = (
            "You are an expert JSSP scheduling analyst. "
            "Explain a fixed already-selected action. "
            "Primary objective context is final makespan (Cmax). "
            "Do not output a new action. "
            "Output format:\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
    elif step_supervision_mode == "action_reason":
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        target_action_reason_text = str(example.get("target_action_reason_text", "")).strip()
        if target_action_reason_text:
            assistant_content = target_action_reason_text
        else:
            assistant_content = f"{target_text}\n{reason_target_text}" if reason_target_text else target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code and explain briefly. "
            "Output format:\n"
            "<Axxxx>\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
        user_content = state_text
    else:
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        assistant_content = target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code. "
            "Always output exactly one code in this format: <Axxxx>."
        )
        user_content = state_text

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]


def _normalize_token_ids(tokenized_output: Any) -> List[int]:
    if tokenized_output is None:
        return []
    if hasattr(tokenized_output, "tolist"):
        tokenized_output = tokenized_output.tolist()
    if isinstance(tokenized_output, tuple):
        tokenized_output = list(tokenized_output)
    if isinstance(tokenized_output, list):
        if tokenized_output and isinstance(tokenized_output[0], list):
            if len(tokenized_output) != 1:
                raise ValueError("Expected a single tokenized sequence, got a batch.")
            tokenized_output = tokenized_output[0]
        return [int(x) for x in tokenized_output]
    raise TypeError(f"Unsupported tokenized output type: {type(tokenized_output)!r}")


def _collect_action_token_ids(tokenizer) -> List[int]:
    cached = getattr(tokenizer, "_cached_action_token_ids", None)
    if cached is not None:
        return list(cached)

    token_ids = []
    seen = set()
    for token in getattr(tokenizer, "additional_special_tokens", []) or []:
        parsed = _normalize_action_code(str(token))
        if parsed is None:
            continue
        try:
            token_id = action_code_to_token_id(tokenizer, str(token))
        except Exception:
            continue
        token_id = int(token_id)
        if token_id in seen:
            continue
        seen.add(token_id)
        token_ids.append(token_id)
    setattr(tokenizer, "_cached_action_token_ids", tuple(token_ids))
    return token_ids


def _find_prompt_token_count(tokenizer, prompt_messages, full_ids: Sequence[int]) -> int:
    prompt_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        )
    )
    if list(full_ids[: len(prompt_ids)]) == prompt_ids:
        return len(prompt_ids)

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    if callable(getattr(tokenizer, "__call__", None)):
        prompt_text_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        prompt_text_ids = [int(x) for x in prompt_text_ids]
        if list(full_ids[: len(prompt_text_ids)]) == prompt_text_ids:
            return len(prompt_text_ids)

    raise ValueError("Could not align prompt and assistant token boundary for step supervision.")


def _extract_action_codes(example: Dict[str, Any]) -> List[str]:
    codes: List[str] = []
    raw_codes = example.get("action_codes")
    if isinstance(raw_codes, list):
        codes.extend(str(code) for code in raw_codes if str(code).strip())
    action_code_to_job = example.get("action_code_to_job")
    if isinstance(action_code_to_job, dict):
        codes.extend(str(code) for code in action_code_to_job.keys() if str(code).strip())
    if not codes:
        raise ValueError(
            "Missing structured action code metadata in step example. "
            "Expected `action_codes` or `action_code_to_job`."
        )
    deduped = []
    seen = set()
    for code in codes:
        parsed = _normalize_action_code(str(code))
        if parsed is None or parsed in seen:
            continue
        seen.add(parsed)
        deduped.append(parsed)
    return deduped


def build_step_supervision_example(
    example: Dict[str, Any],
    tokenizer,
    step_supervision_mode: str = "action_only",
    max_length: int | None = None,
    action_loss_weight: float = 1.0,
):
    messages = build_step_messages(
        example=example,
        step_supervision_mode=step_supervision_mode,
    )
    full_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
    )
    prompt_len = _find_prompt_token_count(tokenizer, messages[:-1], full_ids)
    if prompt_len >= len(full_ids):
        raise ValueError("Assistant span is empty after chat template tokenization.")

    labels = [-100] * len(full_ids)
    loss_weights = [0.0] * len(full_ids)
    attention_mask = [1] * len(full_ids)
    action_target_mask = [0] * len(full_ids)
    assistant_positions = list(range(prompt_len, len(full_ids)))

    action_token_ids = set(_collect_action_token_ids(tokenizer))
    feasible_action_ids = []
    for code in _extract_action_codes(example):
        token_id = tokenizer.convert_tokens_to_ids(str(code))
        if token_id is None:
            continue
        feasible_action_ids.append(int(token_id))
    feasible_action_ids = sorted(set(feasible_action_ids))
    action_position = None
    if step_supervision_mode in {"action_only", "action_reason"}:
        for pos in assistant_positions:
            if int(full_ids[pos]) in action_token_ids:
                action_position = pos
                break
        if action_position is None:
            raise ValueError(
                "Could not find action token inside assistant span. "
                "Check action special-token installation and step targets."
            )

    if step_supervision_mode == "action_only":
        labels[action_position] = int(full_ids[action_position])
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    elif step_supervision_mode == "action_reason":
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    else:
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0

    if max_length is not None and int(max_length) > 0 and len(full_ids) > int(max_length):
        full_ids = full_ids[: int(max_length)]
        attention_mask = attention_mask[: int(max_length)]
        labels = labels[: int(max_length)]
        loss_weights = loss_weights[: int(max_length)]
        action_target_mask = action_target_mask[: int(max_length)]

    supervised_token_count = sum(1 for label in labels if int(label) != -100)
    if supervised_token_count <= 0:
        raise ValueError(
            "No supervised assistant tokens remain after truncation. "
            "Increase max_length or shorten prompts."
        )

    out = dict(example)
    out["text"] = ""
    out["input_ids"] = [int(x) for x in full_ids]
    out["attention_mask"] = [int(x) for x in attention_mask]
    out["labels"] = [int(x) for x in labels]
    out["loss_weights"] = [float(x) for x in loss_weights]
    out["action_target_mask"] = [int(x) for x in action_target_mask]
    out["feasible_action_ids"] = [int(x) for x in feasible_action_ids]
    out["prompt_token_count"] = int(min(prompt_len, len(full_ids)))
    out["assistant_token_count"] = int(max(0, len(full_ids) - min(prompt_len, len(full_ids))))
    out["supervised_token_count"] = int(supervised_token_count)
    return out


CANDIDATE_SCORE_TOKEN_DEFAULT = "<CAND_SCORE>"


def ensure_candidate_score_token(
    tokenizer,
    model=None,
    token: str = CANDIDATE_SCORE_TOKEN_DEFAULT,
):
    token = str(token)
    current_vocab = tokenizer.get_vocab()
    existing_specials = set(getattr(tokenizer, 'additional_special_tokens', []) or [])
    added = 0
    if token not in current_vocab and token not in existing_specials:
        added = int(tokenizer.add_special_tokens({'additional_special_tokens': [token]}))
    if model is not None:
        target_size = len(tokenizer)
        input_size = int(model.get_input_embeddings().weight.shape[0])
        if input_size != target_size:
            model.resize_token_embeddings(target_size)
    token_id = tokenizer.convert_tokens_to_ids(token)
    return {
        'token': token,
        'token_id': int(token_id),
        'num_added_tokens': int(added),
        'vocab_size': int(len(tokenizer)),
    }


def extract_candidate_transition_entries_for_scoring(
    state_text: str,
    feasible_action_codes: Sequence[str],
    code_width: int = 4,
):
    if not isinstance(state_text, str) or not state_text.strip():
        raise ValueError('state_text must be a non-empty string for candidate scoring.')
    feasible_set = {str(code).strip() for code in feasible_action_codes if str(code).strip()}
    if not feasible_set:
        raise ValueError('feasible_action_codes must be non-empty for candidate scoring.')

    rebuilt_lines = []
    candidate_action_codes_in_order = []
    candidate_display_lines_in_order = []
    candidate_original_lines_in_order = []

    candidate_idx = 0
    for line in state_text.splitlines():
        stripped = line.strip()
        parsed_code = parse_action_code(stripped, code_width=code_width)
        if parsed_code is None or parsed_code not in feasible_set or ' | ' not in line:
            rebuilt_lines.append(line)
            continue

        feature_suffix = str(line.split(' | ', 1)[1]).strip()
        candidate_idx += 1
        display_line = f'Candidate {candidate_idx} | {feature_suffix}'
        rebuilt_lines.append(display_line)
        candidate_action_codes_in_order.append(str(parsed_code))
        candidate_display_lines_in_order.append(display_line)
        candidate_original_lines_in_order.append(str(line))

    if not candidate_action_codes_in_order:
        raise ValueError(
            'No candidate transition lines were found for candidate scoring. '
            f'feasible_action_codes={list(feasible_set)[:8]}'
        )

    return {
        'ordinalized_state_text': '\n'.join(rebuilt_lines),
        'candidate_action_codes_in_order': list(candidate_action_codes_in_order),
        'candidate_display_lines_in_order': list(candidate_display_lines_in_order),
        'candidate_original_lines_in_order': list(candidate_original_lines_in_order),
    }


def build_candidate_query_prompt_text(
    tokenizer,
    state_text: str,
    eval_candidate_line: str,
    score_token: str = CANDIDATE_SCORE_TOKEN_DEFAULT,
) -> str:
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an expert JSSP scheduler. '
                'Read the full current step state and the entire feasible candidate set. '
                'The queried candidate is one member of that feasible set. '
                'Represent how strong the queried candidate is relative to the other feasible actions.'
            ),
        },
        {
            'role': 'user',
            'content': (
                f'{state_text}\n\n'
                'Candidate under evaluation:\n'
                f'{str(eval_candidate_line)}\n'
                'Task: Compare the queried candidate against all feasible candidates shown above. '
                'Represent the queried candidate score at the single marker shown below.\n'
                f'{str(score_token)}'
            ),
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def build_candidate_scoring_example(
    example,
    tokenizer,
    max_length,
    score_token: str = CANDIDATE_SCORE_TOKEN_DEFAULT,
):
    state_text = str(example.get('state_text', ''))
    feasible_action_codes = [str(x) for x in example.get('feasible_action_codes', [])]
    target_action_code = str(example.get('target_text', '')).strip()
    if not target_action_code:
        raise ValueError('Missing target_text for candidate scoring example.')

    prepared = extract_candidate_transition_entries_for_scoring(
        state_text=state_text,
        feasible_action_codes=feasible_action_codes,
        code_width=int(example.get('action_code_width', CFG.get('action_code_width', 4))),
    )
    ordinalized_state_text = str(prepared['ordinalized_state_text'])
    candidate_codes_in_order = list(prepared['candidate_action_codes_in_order'])
    candidate_display_lines_in_order = list(prepared['candidate_display_lines_in_order'])
    candidate_original_lines_in_order = list(prepared['candidate_original_lines_in_order'])

    if target_action_code not in candidate_codes_in_order:
        raise ValueError(
            'target_action_code was not found in candidate order derived from state_text. '
            f'target={target_action_code}, candidate_codes={candidate_codes_in_order[:16]}'
        )

    score_token_id = int(tokenizer.convert_tokens_to_ids(score_token))
    if score_token_id < 0:
        raise ValueError(f'score token is not in tokenizer vocab: {score_token}')

    candidate_query_texts = []
    candidate_query_input_ids = []
    candidate_query_attention_masks = []
    candidate_score_marker_positions = []

    for candidate_line in candidate_display_lines_in_order:
        prompt_text = build_candidate_query_prompt_text(
            tokenizer=tokenizer,
            state_text=ordinalized_state_text,
            eval_candidate_line=candidate_line,
            score_token=score_token,
        )
        tokenized = tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=int(max_length),
        )
        input_ids = [int(x) for x in tokenized.get('input_ids', [])]
        attention_mask = [int(x) for x in tokenized.get('attention_mask', [1] * len(input_ids))]
        marker_positions = [
            int(idx) for idx, token_id in enumerate(input_ids)
            if int(token_id) == int(score_token_id)
        ]
        if len(marker_positions) != 1:
            raise ValueError(
                'Expected exactly one candidate score marker per query prompt. '
                f'count={len(marker_positions)}, candidate_line={candidate_line}, max_length={int(max_length)}'
            )
        candidate_query_texts.append(prompt_text)
        candidate_query_input_ids.append(input_ids)
        candidate_query_attention_masks.append(attention_mask)
        candidate_score_marker_positions.append(int(marker_positions[0]))

    out = dict(example)
    out['ordinalized_state_text'] = str(ordinalized_state_text)
    out['candidate_query_texts'] = list(candidate_query_texts)
    out['candidate_query_input_ids'] = list(candidate_query_input_ids)
    out['candidate_query_attention_masks'] = list(candidate_query_attention_masks)
    out['candidate_score_marker_positions'] = list(candidate_score_marker_positions)
    out['candidate_action_codes_in_order'] = list(candidate_codes_in_order)
    out['candidate_display_lines_in_order'] = list(candidate_display_lines_in_order)
    out['candidate_original_lines_in_order'] = list(candidate_original_lines_in_order)
    out['target_candidate_index'] = int(candidate_codes_in_order.index(target_action_code))
    out['prompt_token_count'] = int(sum(len(ids) for ids in candidate_query_input_ids))
    out['assistant_token_count'] = 0
    out['supervised_token_count'] = int(len(candidate_codes_in_order))
    return out


## 3) 학습 실행 (full)


#### 이건 데이터 전처리 및 모델 로딩

In [ ]:
from collections import Counter
import json
import os
import random
import torch
from pathlib import Path
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import TrainerCallback
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainerCallback
base_model_name = MODEL_MAP[CFG['model_type']]
print('base model:', base_model_name)

adapter_role = str(CFG.get('adapter_role', 'policy')).lower()
resolved_step_supervision_mode = {'policy': 'action_only', 'reason': 'reason_only', 'mixed': 'action_reason'}.get(adapter_role)
if resolved_step_supervision_mode is None:
    raise ValueError(f"Unsupported CFG['adapter_role']={adapter_role}")
CFG['step_supervision_mode'] = resolved_step_supervision_mode
if adapter_role == 'policy':
    CFG['hf_model_repo_out'] = CFG.get('policy_hf_model_repo_out', CFG['hf_model_repo_out'])
elif adapter_role == 'reason':
    CFG['hf_model_repo_out'] = CFG.get('reason_hf_model_repo_out', CFG['hf_model_repo_out'])
print('adapter_role:', adapter_role)
print('resolved step supervision mode:', resolved_step_supervision_mode)

task_type = resolve_task_type(CFG)
output_dir = Path(resolve_output_dir(CFG, task_type))
output_dir.mkdir(parents=True, exist_ok=True)

if CFG.get('step_dataset_path'):
    step_dataset_path = os.path.expanduser(CFG['step_dataset_path'])
else:
    env_mode = str(CFG.get('env_mode', 'serial')).lower()
    env_suffix = '_dispatch' if env_mode == 'dispatch' else ''
    if adapter_role == 'reason':
        dataset_repo = CFG.get(f'reason_step_dataset_hf_repo{env_suffix}', CFG.get('reason_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'reason_step_dataset_hf_file{env_suffix}', CFG.get('reason_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'reason_step_dataset_local_path{env_suffix}', CFG.get('reason_step_dataset_local_path', CFG['step_dataset_local_path']))
    elif adapter_role == 'mixed':
        dataset_repo = CFG.get(f'mixed_step_dataset_hf_repo{env_suffix}', CFG.get('mixed_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'mixed_step_dataset_hf_file{env_suffix}', CFG.get('mixed_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'mixed_step_dataset_local_path{env_suffix}', CFG.get('mixed_step_dataset_local_path', CFG['step_dataset_local_path']))
    else:
        dataset_repo = CFG.get(f'policy_step_dataset_hf_repo{env_suffix}', CFG.get('policy_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'policy_step_dataset_hf_file{env_suffix}', CFG.get('policy_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'policy_step_dataset_local_path{env_suffix}', CFG.get('policy_step_dataset_local_path', CFG['step_dataset_local_path']))

    if CFG['dataset_source'] == 'hf':
        step_dataset_path = hf_hub_download(
            repo_id=dataset_repo,
            repo_type='dataset',
            filename=dataset_file,
            token=HF_TOKEN,
        )
    elif CFG['dataset_source'] == 'local':
        step_dataset_path = dataset_local_path
    else:
        raise ValueError("CFG['dataset_source'] must be 'hf' or 'local'")

print('step dataset:', step_dataset_path)

dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16


def _load_unsloth_model_with_chat_template_fallback(**load_kwargs):
    try:
        return FastLanguageModel.from_pretrained(**load_kwargs)
    except Exception as exc:
        if "additional_chat_templates" not in str(exc):
            raise
        retry_kwargs = dict(load_kwargs)
        retry_kwargs["local_files_only"] = True
        print("[Warn] additional_chat_templates 404; retrying with local_files_only=True using cached files.")
        return FastLanguageModel.from_pretrained(**retry_kwargs)

model, tokenizer = _load_unsloth_model_with_chat_template_fallback(
    model_name=base_model_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)

token_install = ensure_action_special_tokens(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
validate_action_tokenizer_installation(
    tokenizer=tokenizer,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
print('action token install:', token_install)
candidate_score_token_install = ensure_candidate_score_token(
    tokenizer=tokenizer,
    model=model,
    token=str(CFG.get('candidate_score_token', '<CAND_SCORE>')),
)
print('candidate score token install:', candidate_score_token_install)
action_row_geometry_before = summarize_action_token_row_geometry(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
)
print('action token row geometry before reinit:', action_row_geometry_before)
action_row_reinit = maybe_reinitialize_action_token_rows(
    tokenizer=tokenizer,
    model=model,
    train_lm_head=bool(CFG.get('train_lm_head', False)),
    train_embed_tokens=bool(CFG.get('train_embed_tokens', False)),
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
    enabled=bool(CFG.get('reinit_action_token_rows_when_frozen', True)),
    scale=float(CFG.get('action_token_reinit_scale', 0.02)),
    seed=CFG.get('action_token_reinit_seed', CFG.get('seed', 42)),
    share_input_output_rows=bool(CFG.get('action_token_reinit_share_input_output_rows', True)),
)
print('action token row reinit:', action_row_reinit)
action_row_geometry_after = summarize_action_token_row_geometry(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
)
print('action token row geometry after reinit:', action_row_geometry_after)

modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
if CFG['train_lm_head']:
    modules.append('lm_head')
if CFG['train_embed_tokens']:
    modules.append('embed_tokens')
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG['lora_r'],
    target_modules=modules,
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    bias=CFG['bias'],
    use_rslora=CFG['use_rslora'],
    use_gradient_checkpointing=CFG['use_gradient_checkpointing'],
    random_state=CFG['random_state'],
    loftq_config=CFG['loftq_config'],
)
print_number_of_trainable_model_parameters(model)


In [ ]:
from datasets import Dataset
import json
import numpy as np
import os
from collections import Counter

REQUIRED_STEP_KEYS = [
    'instance_id',
    'source_index',
    'state_text',
    'reason_input_text',
    'target_text',
    'target_action_reason_text',
    'target_reason_text',
    'reason_target_text',
    'feature_schema_version',
    'num_jobs',
    'num_machines',
    'total_steps',
    'step_idx',
    'action_codes',
    'feasible_action_codes',
    'action_code_to_job',
]


def _normalize_step_row(row):
    out = {}
    out['instance_id'] = str(row.get('instance_id', '') or '')
    source_index = row.get('source_index', -1)
    out['source_index'] = int(source_index) if source_index is not None else -1
    out['state_text'] = str(row.get('state_text', ''))
    out['reason_input_text'] = str(row.get('reason_input_text', ''))
    out['target_text'] = str(row.get('target_text', ''))
    out['target_action_reason_text'] = str(row.get('target_action_reason_text', ''))
    out['target_reason_text'] = str(row.get('target_reason_text', ''))
    out['reason_target_text'] = str(row.get('reason_target_text', ''))
    out['feature_schema_version'] = str(row.get('feature_schema_version', 'unknown'))
    out['num_jobs'] = int(row.get('num_jobs', 0))
    out['num_machines'] = int(row.get('num_machines', 0))
    out['total_steps'] = int(row.get('total_steps', 0))
    out['step_idx'] = int(row.get('step_idx', 0))

    raw_action_codes = row.get('feasible_action_codes')
    if not isinstance(raw_action_codes, list) or not raw_action_codes:
        raise ValueError("Missing non-empty 'feasible_action_codes' in step row.")
    out['action_codes'] = [str(x) for x in raw_action_codes if str(x).strip()]
    if not out['action_codes']:
        raise ValueError("Normalized 'feasible_action_codes' became empty in step row.")
    out['feasible_action_codes'] = list(out['action_codes'])
    out['num_feasible_actions'] = int(len(out['action_codes']))

    raw_action_code_to_job = row.get('action_code_to_job')
    if not isinstance(raw_action_code_to_job, dict) or not raw_action_code_to_job:
        raise ValueError("Missing non-empty 'action_code_to_job' in step row.")
    out['action_code_to_job'] = {str(k): int(v) for k, v in raw_action_code_to_job.items()}
    out['raw_feasible_action_codes'] = list(out['action_codes'])
    return out


def _scan_raw_feasible_stats(path):
    path = os.path.expanduser(str(path))
    total_rows = 0
    feasible_counter = Counter()
    with open(path, 'r', encoding='utf-8') as f:
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                normalized = _normalize_step_row(row)
                total_rows += 1
                feasible_counter[int(normalized.get('num_feasible_actions', 0))] += 1
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                normalized = _normalize_step_row(row)
                total_rows += 1
                feasible_counter[int(normalized.get('num_feasible_actions', 0))] += 1
    return total_rows, feasible_counter


def _iter_step_rows(path, row_cap=None, min_feasible_actions=1):
    path = os.path.expanduser(str(path))
    emitted = 0
    row_cap = None if row_cap is None else int(row_cap)
    min_feasible_actions = max(1, int(min_feasible_actions))
    with open(path, 'r', encoding='utf-8') as f:
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                normalized = _normalize_step_row(row)
                if int(normalized.get('num_feasible_actions', 0)) < min_feasible_actions:
                    continue
                yield normalized
                emitted += 1
                if row_cap is not None and emitted >= row_cap:
                    break
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                normalized = _normalize_step_row(row)
                if int(normalized.get('num_feasible_actions', 0)) < min_feasible_actions:
                    continue
                yield normalized
                emitted += 1
                if row_cap is not None and emitted >= row_cap:
                    break


def _try_get_source_index_array(ds):
    if 'source_index' not in set(ds.column_names):
        return None
    arrow_table = getattr(ds, '_data', None)
    if arrow_table is not None and hasattr(arrow_table, 'column'):
        try:
            column = arrow_table.column('source_index')
            chunks = [
                np.asarray(chunk.to_numpy(zero_copy_only=False), dtype=np.int64)
                for chunk in getattr(column, 'chunks', [])
            ]
            if chunks:
                return np.concatenate(chunks)
        except Exception:
            pass
    return np.asarray(ds['source_index'], dtype=np.int64)


def _try_get_int_array(ds, column_name):
    if column_name not in set(ds.column_names):
        return None
    arrow_table = getattr(ds, '_data', None)
    if arrow_table is not None and hasattr(arrow_table, 'column'):
        try:
            column = arrow_table.column(column_name)
            chunks = [
                np.asarray(chunk.to_numpy(zero_copy_only=False), dtype=np.int64)
                for chunk in getattr(column, 'chunks', [])
            ]
            if chunks:
                return np.concatenate(chunks)
        except Exception:
            pass
    return np.asarray(ds[column_name], dtype=np.int64)


def _resolve_instance_keys(ds):
    columns = set(ds.column_names)
    if 'source_index' in columns:
        source_values = _try_get_source_index_array(ds)
        if source_values is not None:
            return [f"source_{int(x)}" for x in source_values.tolist()]

    instance_values = ds['instance_id'] if 'instance_id' in columns else None
    if instance_values is None:
        raise ValueError("Step dataset must contain either 'instance_id' or 'source_index' for instance-level split.")
    keys = []
    for raw_instance_id in instance_values:
        instance_id = '' if raw_instance_id is None else str(raw_instance_id).strip()
        if not instance_id:
            raise ValueError('Empty instance_id encountered and source_index is unavailable for instance-level split.')
        keys.append(instance_id)
    return keys


def _ordered_unique(keys):
    seen = set()
    ordered = []
    for key in keys:
        if key in seen:
            continue
        seen.add(key)
        ordered.append(key)
    return ordered


def _split_indices_by_instance(
    ds,
    test_ratio,
    split_seed,
    enable_eval_split,
    split_mode='fixed_per_size',
    eval_instances_per_size=3,
):
    source_values = _try_get_source_index_array(ds)
    if source_values is not None and source_values.size > 0:
        ordered_pos = np.sort(np.unique(source_values, return_index=True)[1])
        ordered_sources = source_values[ordered_pos].tolist()
        ordered_instances = [f"source_{int(x)}" for x in ordered_sources]
        if not enable_eval_split:
            train_indices = np.arange(len(source_values), dtype=np.int64).tolist()
            return train_indices, [], ordered_instances, []

        resolved_split_mode = str(split_mode).lower()
        rng = random.Random(int(split_seed))
        if resolved_split_mode == 'fixed_per_size':
            num_jobs_values = _try_get_int_array(ds, 'num_jobs')
            num_machines_values = _try_get_int_array(ds, 'num_machines')
            if num_jobs_values is None or num_machines_values is None:
                raise ValueError("fixed_per_size eval split requires 'num_jobs' and 'num_machines' columns.")
            ordered_num_jobs = num_jobs_values[ordered_pos].tolist()
            ordered_num_machines = num_machines_values[ordered_pos].tolist()
            size_to_sources = {}
            ordered_sizes = []
            for source_idx, n_jobs, n_machines in zip(ordered_sources, ordered_num_jobs, ordered_num_machines):
                size_key = (int(n_jobs), int(n_machines))
                if size_key not in size_to_sources:
                    size_to_sources[size_key] = []
                    ordered_sizes.append(size_key)
                size_to_sources[size_key].append(int(source_idx))
            eval_sources = []
            per_size = max(1, int(eval_instances_per_size))
            for size_key in ordered_sizes:
                candidates = list(size_to_sources[size_key])
                rng.shuffle(candidates)
                eval_sources.extend(candidates[: min(per_size, len(candidates))])
        else:
            shuffled_sources = ordered_sources[:]
            rng.shuffle(shuffled_sources)
            eval_instance_count = max(1, int(round(len(shuffled_sources) * float(test_ratio))))
            eval_instance_count = min(eval_instance_count, max(1, len(shuffled_sources) - 1))
            eval_sources = shuffled_sources[:eval_instance_count]

        eval_source_set = {int(x) for x in eval_sources}
        eval_mask = np.isin(source_values, np.asarray(eval_sources, dtype=np.int64))
        train_indices = np.flatnonzero(~eval_mask).astype(np.int64).tolist()
        eval_indices = np.flatnonzero(eval_mask).astype(np.int64).tolist()
        train_instance_ids = [f"source_{int(x)}" for x in ordered_sources if int(x) not in eval_source_set]
        eval_instance_ids = [f"source_{int(x)}" for x in ordered_sources if int(x) in eval_source_set]
        return train_indices, eval_indices, train_instance_ids, eval_instance_ids

    instance_keys = _resolve_instance_keys(ds)
    ordered_instances = _ordered_unique(instance_keys)
    if not ordered_instances:
        raise ValueError('No instances found for instance-level split.')

    if not enable_eval_split:
        return list(range(len(ds))), [], ordered_instances, []

    shuffled_instances = ordered_instances[:]
    rng = random.Random(int(split_seed))
    rng.shuffle(shuffled_instances)
    eval_instance_count = max(1, int(round(len(shuffled_instances) * float(test_ratio))))
    eval_instance_count = min(eval_instance_count, max(1, len(shuffled_instances) - 1))
    eval_instance_set = set(shuffled_instances[:eval_instance_count])

    train_indices = []
    eval_indices = []
    for row_idx, instance_key in enumerate(instance_keys):
        if instance_key in eval_instance_set:
            eval_indices.append(row_idx)
        else:
            train_indices.append(row_idx)

    train_instance_ids = [iid for iid in ordered_instances if iid not in eval_instance_set]
    eval_instance_ids = [iid for iid in ordered_instances if iid in eval_instance_set]
    return train_indices, eval_indices, train_instance_ids, eval_instance_ids


def _filter_by_min_feasible_actions(ds, min_count, split_name):
    min_count = max(1, int(min_count))
    if ds is None or min_count <= 1:
        return ds
    before_count = len(ds)
    ds = ds.filter(
        lambda example: int(example.get('num_feasible_actions', len(example.get('action_codes', [])))) >= min_count,
        num_proc=max(1, int(CFG.get('dataset_num_proc', 1))),
    )
    after_count = len(ds)
    print(f'{split_name} feasible-action filter: min_feasible_actions>={min_count}, kept={after_count:,}/{before_count:,}')
    return ds


enable_eval = bool(CFG.get('enable_eval', True))
train_min_feasible_actions = max(1, int(CFG.get('min_train_feasible_actions', 1)))
eval_min_feasible_actions = max(1, int(CFG.get('min_eval_feasible_actions', 1)))
raw_train_filter_applied = False
raw_load_cap = None
raw_load_min_feasible_actions = 1
if not enable_eval:
    raw_load_min_feasible_actions = train_min_feasible_actions
    raw_train_filter_applied = raw_load_min_feasible_actions > 1
    if raw_train_filter_applied:
        print(f'raw train feasible filter enabled (no eval): min_feasible_actions>={raw_load_min_feasible_actions}')
if (not enable_eval) and CFG.get('max_train_samples') is not None:
    raw_load_cap = int(CFG['max_train_samples'])
    print(f'raw dataset load cap enabled (no eval): {raw_load_cap:,}')

raw_source_total_rows = None
raw_source_feasible_counter = None
# if raw_train_filter_applied:
#     print('scanning original raw dataset before feasible filter ...')
#     raw_source_total_rows, raw_source_feasible_counter = _scan_raw_feasible_stats(step_dataset_path)
#     raw_source_ge2 = sum(v for k, v in raw_source_feasible_counter.items() if int(k) >= 2)
#     raw_source_ge3 = sum(v for k, v in raw_source_feasible_counter.items() if int(k) >= 3)
#     print(f'original raw dataset rows (before feasible filter): {raw_source_total_rows:,}')
#     print('original raw feasible_count_dist_top:', raw_source_feasible_counter.most_common(20))
#     print('original raw feasible_count>=2 rows:', raw_source_ge2)
#     print('original raw feasible_count>=2 ratio:', raw_source_ge2 / max(1, raw_source_total_rows))
#     print('original raw feasible_count>=3 rows:', raw_source_ge3)
#     print('original raw feasible_count>=3 ratio:', raw_source_ge3 / max(1, raw_source_total_rows))

dataset = Dataset.from_generator(
    _iter_step_rows,
    gen_kwargs={
        'path': step_dataset_path,
        'row_cap': raw_load_cap,
        'min_feasible_actions': raw_load_min_feasible_actions,
    },
)
print(f'raw dataset rows: {len(dataset):,}')

# raw_feasible_counter = Counter(int(x) for x in dataset['num_feasible_actions'])
# raw_ge2 = sum(v for k, v in raw_feasible_counter.items() if int(k) >= 2)
# raw_ge3 = sum(v for k, v in raw_feasible_counter.items() if int(k) >= 3)
# print('raw feasible_count_dist_top:', raw_feasible_counter.most_common(20))
# print('raw feasible_count>=2 rows:', raw_ge2)
# print('raw feasible_count>=2 ratio:', raw_ge2 / max(1, len(dataset)))
# print('raw feasible_count>=3 rows:', raw_ge3)
# print('raw feasible_count>=3 ratio:', raw_ge3 / max(1, len(dataset)))
# if raw_source_total_rows is not None:
#     print('raw feasible filter kept rows:', f"{len(dataset):,}/{raw_source_total_rows:,}")
#     print('raw feasible filter kept ratio:', len(dataset) / max(1, raw_source_total_rows))
#     print('raw feasible filter dropped rows:', raw_source_total_rows - len(dataset))
#     print('raw feasible filter dropped ratio:', (raw_source_total_rows - len(dataset)) / max(1, raw_source_total_rows))

test_size = max(0.0, min(0.99, float(CFG.get('eval_split_ratio', 0.05))))
train_indices, eval_indices, train_instance_ids, eval_instance_ids = _split_indices_by_instance(
    dataset,
    test_ratio=test_size,
    split_seed=CFG['seed'],
    enable_eval_split=enable_eval,
    split_mode=CFG.get('eval_split_mode', 'fixed_per_size'),
    eval_instances_per_size=CFG.get('eval_instances_per_size', 3),
)

train_dataset = dataset.select(train_indices)
eval_dataset = dataset.select(eval_indices) if enable_eval else None

overlap_count = len(set(train_instance_ids) & set(eval_instance_ids))
print(
    'instance split:',
    f"mode={CFG.get('eval_split_mode', 'fixed_per_size')}",
    f"train_instances={len(train_instance_ids):,}",
    f"eval_instances={len(eval_instance_ids):,}",
    f"overlap={overlap_count}",
)
print(
    'row counts before map:',
    f"train_rows={len(train_dataset):,}",
    f"eval_rows={(len(eval_dataset) if eval_dataset is not None else 0):,}",
)

# train_feasible_counter_before_filter = Counter(int(x) for x in train_dataset['num_feasible_actions'])
# train_ge2_before_filter = sum(v for k, v in train_feasible_counter_before_filter.items() if int(k) >= 2)
# train_ge3_before_filter = sum(v for k, v in train_feasible_counter_before_filter.items() if int(k) >= 3)
# print('train feasible_count_dist_top (before feasible filter):', train_feasible_counter_before_filter.most_common(20))
# print('train feasible_count>=2 rows (before feasible filter):', train_ge2_before_filter)
# print('train feasible_count>=2 ratio (before feasible filter):', train_ge2_before_filter / max(1, len(train_dataset)))
# print('train feasible_count>=3 rows (before feasible filter):', train_ge3_before_filter)
# print('train feasible_count>=3 ratio (before feasible filter):', train_ge3_before_filter / max(1, len(train_dataset)))

if raw_train_filter_applied:
    print(f'train feasible-action filter already applied at raw load stage: min_feasible_actions>={train_min_feasible_actions}, rows={len(train_dataset):,}')
elif train_min_feasible_actions > 1:
    train_dataset = _filter_by_min_feasible_actions(train_dataset, train_min_feasible_actions, 'train')

if eval_dataset is not None and eval_min_feasible_actions > 1:
    eval_dataset = _filter_by_min_feasible_actions(eval_dataset, eval_min_feasible_actions, 'eval')

if CFG['shuffle_data']:
    print(f"shuffle train split enabled (seed={CFG['shuffle_seed']})")
    train_dataset = train_dataset.shuffle(seed=CFG['shuffle_seed'])
else:
    print('shuffle train split disabled')

if CFG.get('max_train_samples') is not None and raw_load_cap is None:
    train_cap = min(int(CFG['max_train_samples']), len(train_dataset))
    print('train sample cap:', train_cap)
    train_dataset = train_dataset.select(range(train_cap))
elif CFG.get('max_train_samples') is not None:
    print('train sample cap already applied at raw load stage:', len(train_dataset))

if eval_dataset is not None and CFG.get('max_eval_samples') is not None:
    eval_cap = min(int(CFG['max_eval_samples']), len(eval_dataset))
    print('eval sample cap:', eval_cap)
    eval_dataset = eval_dataset.shuffle(seed=int(CFG.get('eval_subset_seed', 42))).select(range(eval_cap))


policy_head_type = str(CFG.get('policy_head_type', 'token_generation')).lower()
if policy_head_type == 'candidate_scoring':
    if resolved_step_supervision_mode != 'action_only':
        raise ValueError(
            f"candidate_scoring currently supports policy/action_only only, got mode={resolved_step_supervision_mode}"
        )
    fmt = partial(
        build_candidate_scoring_example,
        tokenizer=tokenizer,
        max_length=CFG['max_seq_length'],
        score_token=str(CFG.get('candidate_score_token', '<CAND_SCORE>')),
    )
    print('dataset formatter:', 'candidate_scoring')
else:
    fmt = partial(
        build_step_supervision_example,
        tokenizer=tokenizer,
        step_supervision_mode=resolved_step_supervision_mode,
        max_length=CFG['max_seq_length'],
        action_loss_weight=float(CFG.get('action_loss_weight', 4.0)),
    )
    print('dataset formatter:', 'token_generation')
step_map_num_proc = max(1, int(CFG.get('dataset_num_proc', 1)))
print('step supervision map num_proc:', step_map_num_proc)
train_dataset = train_dataset.map(fmt, num_proc=step_map_num_proc)
if eval_dataset is not None:
    eval_dataset = eval_dataset.map(fmt, num_proc=step_map_num_proc)

from collections import Counter


def _validate_action_supervision_rows(ds, label, mode, sample_limit=512):
    if ds is None or len(ds) == 0:
        print(f'{label} supervision check skipped: empty dataset')
        return
    if mode not in {'action_only', 'action_reason'}:
        print(f'{label} supervision check skipped for mode={mode}')
        return

    limit = min(int(sample_limit), len(ds))
    feasible_len_counter = Counter()
    matched = 0
    for i in range(limit):
        row = ds[i]
        action_positions = [idx for idx, flag in enumerate(row.get('action_target_mask', [])) if int(flag) == 1]
        if len(action_positions) != 1:
            raise ValueError(f'{label} row {i}: expected exactly one action target position, got {len(action_positions)}')
        pos = int(action_positions[0])
        labels_row = row.get('labels', [])
        feasible_ids = [int(x) for x in row.get('feasible_action_ids', [])]
        if pos >= len(labels_row):
            raise ValueError(f'{label} row {i}: action position {pos} exceeds labels length {len(labels_row)}')
        target_id = int(labels_row[pos])
        if target_id == -100:
            raise ValueError(f'{label} row {i}: action target label is -100 at position {pos}')
        if target_id not in feasible_ids:
            raise ValueError(
                f'{label} row {i}: target_id {target_id} not found in feasible_action_ids '
                f'(count={len(feasible_ids)}, feasible_head={feasible_ids[:8]})'
            )
        feasible_len_counter[len(feasible_ids)] += 1
        matched += 1

    print(f'{label} action supervision check:', {
        'checked_rows': limit,
        'matched_targets': matched,
        'feasible_count_dist_top': feasible_len_counter.most_common(10),
        'single_feasible_ratio': (feasible_len_counter.get(1, 0) / limit) if limit else 0.0,
    })


# _validate_action_supervision_rows(train_dataset, 'train', resolved_step_supervision_mode)
# if eval_dataset is not None:
#     _validate_action_supervision_rows(eval_dataset, 'eval', resolved_step_supervision_mode)

enable_eval = bool(CFG.get('enable_eval', True))
eval_strategy_value = CFG['evaluation_strategy'] if enable_eval else 'no'
eval_steps_value = CFG['eval_steps'] if enable_eval else None
eval_dataset_for_trainer = eval_dataset if enable_eval else None

print('train rows:', len(train_dataset))
print('eval rows:', len(eval_dataset) if eval_dataset is not None else 0)
# if len(train_dataset):
#     sample_feature_schema = train_dataset[0].get('feature_schema_version', 'unknown') if hasattr(train_dataset[0], 'get') else 'unknown'
#     print('feature schema:', sample_feature_schema)
#     print('sample prompt\n', train_dataset[0]['text'][:500])
#     print('sample supervision stats', {
#         'prompt_tokens': train_dataset[0].get('prompt_token_count'),
#         'assistant_tokens': train_dataset[0].get('assistant_token_count'),
#         'supervised_tokens': train_dataset[0].get('supervised_token_count'),
#     })

# if len(train_dataset):
#     sample_row = train_dataset[0]
#     print('sample row keys:', list(sample_row.keys()))
#     print('\n[sample state_text]')
#     print(str(sample_row.get('state_text', ''))[:3000])
#     print('\n[sample target_text]')
#     print(sample_row.get('target_text', ''))
#     print('\n[sample input_ids length]')
#     print(len(sample_row.get('input_ids', [])))
#     label_positions = [i for i, x in enumerate(sample_row.get('labels', [])) if int(x) != -100]
#     print('\n[sample labels non -100 positions]')
#     print(label_positions[:50], '... total =', len(label_positions))
#     action_positions = [i for i, x in enumerate(sample_row.get('action_target_mask', [])) if int(x) == 1]
#     print('\n[sample action_target_mask positions]')
#     print(action_positions)
#     print('\n[sample action_codes]')
#     print(sample_row.get('action_codes', []))
#     print('\n[sample raw_feasible_action_codes]')
#     print(sample_row.get('raw_feasible_action_codes', []))
#     print('\n[sample feasible_action_ids]')
#     print(sample_row.get('feasible_action_ids', []))
#     print('\n[sample feasible_action_tokens]')
#     print([tokenizer.decode([int(x)], skip_special_tokens=False) for x in sample_row.get('feasible_action_ids', [])])
#     if action_positions:
#         pos = int(action_positions[0])
#         label_id = int(sample_row['labels'][pos])
#         print('\n[sample action target detail]')
#         print({'pos': pos, 'label_id': label_id, 'decoded': tokenizer.decode([label_id], skip_special_tokens=False)})
#         print('target_in_feasible =', label_id in [int(x) for x in sample_row.get('feasible_action_ids', [])])
#         print('sample action loss_weight =', float(sample_row.get('loss_weights', [0.0] * (pos + 1))[pos]))
#     print('\n[sample nonzero loss_weights positions]')
#     nz_weights = [(i, float(x)) for i, x in enumerate(sample_row.get('loss_weights', [])) if float(x) != 0.0]
#     print(nz_weights[:20], '... total =', len(nz_weights))
#     print('\n[sample prompt/assistant/supervised counts]')
#     print({
#         'prompt_token_count': sample_row.get('prompt_token_count'),
#         'assistant_token_count': swjample_row.get('assistant_token_count'),
#         'supervised_token_count': sample_row.get('supervised_token_count'),
#     })

print('dataset preprocessing ready; run the next cell to start training')

#### 이거는 데이터 max_length를 위한 테스트용

In [ ]:
import random

print('[dataset length scan start]')
print('train rows:', len(train_dataset))

# 1. 필요한 column만 먼저 읽어서 10x10 first-step index를 찾음
num_jobs_col = train_dataset['num_jobs']
num_machines_col = train_dataset['num_machines']
step_idx_col = train_dataset['step_idx']

target_indices = [
    i
    for i, (num_jobs, num_machines, step_idx) in enumerate(
        zip(num_jobs_col, num_machines_col, step_idx_col)
    )
    if int(num_jobs) == 10 and int(num_machines) == 10 and int(step_idx) == 0
]

print('10x10 first-step rows:', len(target_indices))

if not target_indices:
    print('No 10x10 first-step rows found.')
else:
    # 2. 정확한 max를 보고 싶으면 full_scan=True
    #    대충 감만 보면 full_scan=False + sample_size 사용
    full_scan = True
    sample_size = 500

    if full_scan or len(target_indices) <= sample_size:
        scan_indices = target_indices
        print('scan mode: full')
    else:
        scan_indices = random.sample(target_indices, sample_size)
        print('scan mode: sample')
        print('sample_size:', len(scan_indices))

    longest_row = None
    seq_lens = []

    for row_idx in scan_indices:
        row = train_dataset[row_idx]
        item = {
            'row_idx': int(row_idx),
            'instance_id': str(row.get('instance_id', '')),
            'num_jobs': int(row.get('num_jobs', 0) or 0),
            'num_machines': int(row.get('num_machines', 0) or 0),
            'step_idx': int(row.get('step_idx', -1) or -1),
            'seq_len': int(len(row.get('input_ids', []))),
            'prompt_tokens': int(row.get('prompt_token_count', 0) or 0),
            'assistant_tokens': int(row.get('assistant_token_count', 0) or 0),
            'supervised_tokens': int(row.get('supervised_token_count', 0) or 0),
            'feasible_actions': int(row.get('num_feasible_actions', len(row.get('action_codes', [])))),
        }
        seq_lens.append(item['seq_len'])
        if longest_row is None or item['seq_len'] > longest_row['seq_len']:
            longest_row = item

    print('\n[longest 10x10 first-step sequence]')
    print(longest_row)
    print('recommended_max_seq_length_min:', longest_row['seq_len'] + 32)

    print('\n[length summary]')
    print({
        'count': len(seq_lens),
        'min': min(seq_lens),
        'max': max(seq_lens),
        'avg': round(sum(seq_lens) / len(seq_lens), 2),
    })



#### 이게 학습

In [ ]:

print('starting training cell; re-run previous cell if you changed dataset/preprocessing settings or want a fresh model state')
import inspect
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer as HFTrainer, TrainingArguments, TrainerCallback

policy_head_type = str(CFG.get('policy_head_type', 'token_generation')).lower()
if policy_head_type != 'candidate_scoring':
    raise ValueError(f"[fix] notebook is configured for candidate_scoring, got policy_head_type={policy_head_type}")
if str(CFG.get('adapter_role', 'policy')).lower() != 'policy':
    raise ValueError('[fix] candidate_scoring path currently supports adapter_role=policy only.')
if str(CFG.get('step_supervision_mode', 'action_only')).lower() != 'action_only':
    raise ValueError('[fix] candidate_scoring path currently supports step_supervision_mode=action_only only.')

print('candidate query reranker training enabled')
print('score token:', str(CFG.get('candidate_score_token', '<CAND_SCORE>')))
print('query forward batch size:', int(CFG.get('candidate_scoring_query_forward_batch_size', 16)))

class CandidateScoringCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, features):
        flat_input_ids = []
        flat_attention_mask = []
        flat_score_marker_positions = []
        candidate_counts = []
        target_candidate_index = []
        candidate_action_codes = []
        for feature in features:
            query_ids = [list(x) for x in feature.get('candidate_query_input_ids', [])]
            query_masks = [list(x) for x in feature.get('candidate_query_attention_masks', [])]
            query_marker_positions = [int(x) for x in feature.get('candidate_score_marker_positions', [])]
            query_codes = list(feature.get('candidate_action_codes_in_order', []))
            count = int(len(query_ids))
            if count <= 0:
                raise ValueError('candidate_query_input_ids must be non-empty for candidate scoring.')
            if not (len(query_masks) == count == len(query_marker_positions) == len(query_codes)):
                raise ValueError(
                    'Candidate query field lengths mismatch: '
                    f'ids={len(query_ids)}, masks={len(query_masks)}, markers={len(query_marker_positions)}, codes={len(query_codes)}'
                )
            candidate_counts.append(count)
            target_candidate_index.append(int(feature['target_candidate_index']))
            candidate_action_codes.append(query_codes)
            flat_input_ids.extend(query_ids)
            flat_attention_mask.extend(query_masks)
            flat_score_marker_positions.extend(query_marker_positions)

        max_len = max(len(ids) for ids in flat_input_ids)
        padded_input_ids = []
        padded_attention_mask = []
        for ids, mask in zip(flat_input_ids, flat_attention_mask):
            pad_len = max_len - len(ids)
            padded_input_ids.append(list(ids) + [int(self.pad_token_id)] * pad_len)
            padded_attention_mask.append(list(mask) + [0] * pad_len)

        return {
            'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
            'score_marker_positions': torch.tensor(flat_score_marker_positions, dtype=torch.long),
            'candidate_counts': torch.tensor(candidate_counts, dtype=torch.long),
            'target_candidate_index': torch.tensor(target_candidate_index, dtype=torch.long),
            'candidate_action_codes': candidate_action_codes,
        }


def _try_forward_last_hidden_state(module, *, input_ids, attention_mask):
    outputs = module(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True,
    )
    last_hidden = getattr(outputs, "last_hidden_state", None)
    if last_hidden is not None:
        return last_hidden
    if isinstance(outputs, tuple) and len(outputs) > 0 and torch.is_tensor(outputs[0]):
        return outputs[0]
    raise RuntimeError("last_hidden_state is unavailable from candidate-scoring backbone forward.")


def _forward_final_hidden_state(model, *, input_ids, attention_mask):
    candidate_modules = []

    def _append_module(module):
        if isinstance(module, nn.Module):
            candidate_modules.append(module)

    _append_module(getattr(model, "model", None))
    base_model = getattr(model, "base_model", None)
    _append_module(getattr(base_model, "model", None))
    _append_module(base_model)
    _append_module(getattr(model, "backbone", None))
    _append_module(model)

    seen = set()
    for module in candidate_modules:
        module_id = id(module)
        if module_id in seen:
            continue
        seen.add(module_id)
        try:
            return _try_forward_last_hidden_state(
                module,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
        except Exception:
            continue

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )
    hidden_states = getattr(outputs, "hidden_states", None)
    if hidden_states is None:
        raise RuntimeError("Unable to obtain final hidden state for candidate scoring.")
    return hidden_states[-1]


class CandidateScoringModel(nn.Module):
    def __init__(self, backbone_model, score_token: str, head_init_std: float = 0.02, query_forward_batch_size: int = 16):
        super().__init__()
        self.backbone = backbone_model
        hidden_size = int(getattr(backbone_model.config, 'hidden_size'))
        backbone_param = next(backbone_model.parameters())
        head_device = backbone_param.device
        head_dtype = backbone_param.dtype
        self.candidate_score_head = nn.Linear(hidden_size, 1, bias=True, device=head_device, dtype=head_dtype)
        nn.init.normal_(self.candidate_score_head.weight, mean=0.0, std=float(head_init_std))
        nn.init.zeros_(self.candidate_score_head.bias)
        self.score_token = str(score_token)
        self.policy_head_type = 'candidate_scoring'
        self.query_forward_batch_size = max(1, int(query_forward_batch_size))
        self.config = getattr(backbone_model, 'config', None)

    def _score_flat_queries(self, input_ids, attention_mask, score_marker_positions):
        flat_scores = []
        total_queries = int(input_ids.size(0))
        for start in range(0, total_queries, int(self.query_forward_batch_size)):
            end = min(total_queries, start + int(self.query_forward_batch_size))
            chunk_input_ids = input_ids[start:end]
            chunk_attention_mask = attention_mask[start:end]
            chunk_positions = score_marker_positions[start:end]
            hidden_states = _forward_final_hidden_state(
                self.backbone,
                input_ids=chunk_input_ids,
                attention_mask=chunk_attention_mask,
            )
            row_indices = torch.arange(hidden_states.size(0), device=hidden_states.device)
            gathered = hidden_states[row_indices, chunk_positions.to(hidden_states.device).long()]
            gathered = gathered.to(dtype=self.candidate_score_head.weight.dtype)
            chunk_scores = self.candidate_score_head(gathered).squeeze(-1)
            flat_scores.append(chunk_scores)
            del hidden_states, row_indices, gathered, chunk_scores
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return torch.cat(flat_scores, dim=0)

    def forward(
        self,
        input_ids,
        attention_mask,
        score_marker_positions,
        candidate_counts,
        target_candidate_index=None,
        **kwargs,
    ):
        flat_scores = self._score_flat_queries(
            input_ids=input_ids,
            attention_mask=attention_mask,
            score_marker_positions=score_marker_positions,
        )
        counts = [int(x) for x in candidate_counts.tolist()]
        if sum(counts) != int(flat_scores.shape[0]):
            raise ValueError(
                f'candidate_counts sum mismatch: counts_sum={sum(counts)}, flat_scores={int(flat_scores.shape[0])}'
            )
        max_candidates = max(counts)
        pad_value = torch.finfo(flat_scores.dtype).min
        candidate_scores = flat_scores.new_full((len(counts), max_candidates), pad_value)
        offset = 0
        for row_idx, count in enumerate(counts):
            candidate_scores[row_idx, :count] = flat_scores[offset:offset + count]
            offset += count
        loss = None
        if target_candidate_index is not None:
            loss = F.cross_entropy(candidate_scores.float(), target_candidate_index.long())
        return {
            'loss': loss,
            'candidate_scores': candidate_scores,
            'flat_scores': flat_scores,
        }

    def save_checkpoint_bundle(self, save_dir):
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        self.backbone.save_pretrained(str(save_dir))
        torch.save(
            {
                'state_dict': self.candidate_score_head.state_dict(),
                'score_token': self.score_token,
                'policy_head_type': self.policy_head_type,
                'query_forward_batch_size': int(self.query_forward_batch_size),
            },
            save_dir / 'candidate_scorer.pt',
        )


class CandidateScoringTrainer(HFTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        model_inputs = dict(inputs)
        target_candidate_index = model_inputs.pop('target_candidate_index')
        model_inputs.pop('candidate_action_codes', None)
        outputs = model(**model_inputs, target_candidate_index=target_candidate_index)
        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

    def save_model(self, output_dir=None, _internal_call=False):
        save_dir = Path(output_dir or self.args.output_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_checkpoint_bundle(save_dir)
        tok = getattr(self, 'tokenizer', None)
        if tok is not None:
            tok.save_pretrained(str(save_dir))


def _infer_model_device(model):
    try:
        emb = model.backbone.get_input_embeddings()
        if emb is not None and hasattr(emb, 'weight'):
            return emb.weight.device
    except Exception:
        pass
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device('cpu')


def _print_candidate_scoring_probe(model, batch, max_rows=8, topk=5, prefix='CANDIDATE PROBE'):
    model_device = _infer_model_device(model)
    device_batch = {k: (v.to(model_device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    model_inputs = {k: v for k, v in device_batch.items() if k not in {'target_candidate_index', 'candidate_action_codes'}}
    target_candidate_index = device_batch['target_candidate_index']
    candidate_action_codes = batch['candidate_action_codes']

    was_training = bool(model.training)
    model.eval()
    try:
        with torch.no_grad():
            outputs = model(**model_inputs)
            candidate_scores = outputs['candidate_scores']
            probs = torch.softmax(candidate_scores.float(), dim=-1)
            print(f'\n[{prefix}]')
            shown_rows = min(int(max_rows), int(candidate_scores.shape[0]))
            target_probs_summary = []
            uniform_probs_summary = []
            row_ce_summary = []
            random_ce_summary = []
            top1_hits = []
            for row_idx in range(shown_rows):
                row_codes = list(candidate_action_codes[row_idx])
                row_probs = probs[row_idx, : len(row_codes)]
                row_scores = candidate_scores[row_idx, : len(row_codes)].float()
                target_idx = int(target_candidate_index[row_idx].item())
                target_prob = float(row_probs[target_idx].detach().cpu().item())
                uniform_prob = 1.0 / max(1, len(row_codes))
                entropy = float((-(row_probs * row_probs.clamp_min(1e-12).log())).sum().detach().cpu().item())
                random_ce = float(torch.log(torch.tensor(float(len(row_codes)))).item()) if len(row_codes) > 0 else 0.0
                row_ce = float((-row_probs[target_idx].clamp_min(1e-12).log()).detach().cpu().item())
                topk_row = min(int(topk), len(row_codes))
                top_vals, top_idx = torch.topk(row_probs, k=topk_row)
                top1_idx = int(torch.argmax(row_probs).item())
                target_rank = 1 + int((row_probs > row_probs[target_idx]).sum().item())
                top1_hit = int(top1_idx == target_idx)
                target_probs_summary.append(target_prob)
                uniform_probs_summary.append(uniform_prob)
                row_ce_summary.append(row_ce)
                random_ce_summary.append(random_ce)
                top1_hits.append(top1_hit)
                print({
                    'row_idx': int(row_idx),
                    'feasible_count': int(len(row_codes)),
                    'target_idx': int(target_idx),
                    'target_rank': int(target_rank),
                    'target_action_code': str(row_codes[target_idx]),
                    'top1_action_code': str(row_codes[top1_idx]),
                    'top1_hit': int(top1_hit),
                    'target_prob': float(target_prob),
                    'uniform_prob': float(uniform_prob),
                    'target_prob_delta_vs_uniform': float(target_prob - uniform_prob),
                    'row_ce': float(row_ce),
                    'random_ce': float(random_ce),
                    'score_min': float(row_scores.min().detach().cpu().item()),
                    'score_max': float(row_scores.max().detach().cpu().item()),
                    'score_std': float(row_scores.std().detach().cpu().item()) if len(row_codes) > 1 else 0.0,
                    'entropy': float(entropy),
                    'top_probs': [
                        (str(row_codes[int(j)]), float(p))
                        for p, j in zip(top_vals.tolist(), top_idx.tolist())
                    ],
                })
            if shown_rows > 0:
                print({
                    'summary_rows': int(shown_rows),
                    'avg_target_prob': float(sum(target_probs_summary) / max(1, len(target_probs_summary))),
                    'avg_uniform_prob': float(sum(uniform_probs_summary) / max(1, len(uniform_probs_summary))),
                    'avg_target_prob_delta_vs_uniform': float((sum(target_probs_summary) - sum(uniform_probs_summary)) / max(1, len(target_probs_summary))),
                    'avg_row_ce': float(sum(row_ce_summary) / max(1, len(row_ce_summary))),
                    'avg_random_ce': float(sum(random_ce_summary) / max(1, len(random_ce_summary))),
                    'top1_acc': float(sum(top1_hits) / max(1, len(top1_hits))),
                })
    finally:
        if was_training:
            model.train()


def _print_worst_candidate_scoring_example(model, features, batch, topk=5):
    if not features:
        return
    model_device = _infer_model_device(model)
    device_batch = {k: (v.to(model_device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    model_inputs = {k: v for k, v in device_batch.items() if k not in {'target_candidate_index', 'candidate_action_codes'}}
    target_candidate_index = device_batch['target_candidate_index']
    was_training = bool(model.training)
    model.eval()
    try:
        with torch.no_grad():
            outputs = model(**model_inputs)
            candidate_scores = outputs['candidate_scores']
            probs = torch.softmax(candidate_scores.float(), dim=-1)
            worst_row_idx = 0
            worst_row_ce = None
            for row_idx in range(min(len(features), int(candidate_scores.shape[0]))):
                row_codes = list(batch['candidate_action_codes'][row_idx])
                row_probs = probs[row_idx, : len(row_codes)]
                target_idx = int(target_candidate_index[row_idx].item())
                row_ce = float((-row_probs[target_idx].clamp_min(1e-12).log()).detach().cpu().item())
                if worst_row_ce is None or row_ce > worst_row_ce:
                    worst_row_ce = row_ce
                    worst_row_idx = int(row_idx)

            feature = features[worst_row_idx]
            row_codes = list(batch['candidate_action_codes'][worst_row_idx])
            row_probs = probs[worst_row_idx, : len(row_codes)]
            row_scores = candidate_scores[worst_row_idx, : len(row_codes)].float()
            target_idx = int(target_candidate_index[worst_row_idx].item())
            top1_idx = int(torch.argmax(row_probs).item())
            topk_row = min(int(topk), len(row_codes))
            top_vals, top_idx = torch.topk(row_probs, k=topk_row)

            print('\n[DEBUG worst candidate scoring example]')
            print({
                'worst_row_idx': int(worst_row_idx),
                'criterion': 'max_row_ce',
                'target_idx': int(target_idx),
                'target_action_code': str(row_codes[target_idx]),
                'target_display_line': str(feature.get('candidate_display_lines_in_order', [])[target_idx]) if feature.get('candidate_display_lines_in_order') else '',
                'target_original_line': str(feature.get('candidate_original_lines_in_order', [])[target_idx]) if feature.get('candidate_original_lines_in_order') else '',
                'top1_idx': int(top1_idx),
                'top1_action_code': str(row_codes[top1_idx]),
                'top1_prob': float(row_probs[top1_idx].detach().cpu().item()),
                'target_prob': float(row_probs[target_idx].detach().cpu().item()),
                'row_ce': float(worst_row_ce if worst_row_ce is not None else 0.0),
                'score_min': float(row_scores.min().detach().cpu().item()),
                'score_max': float(row_scores.max().detach().cpu().item()),
                'score_std': float(row_scores.std().detach().cpu().item()) if len(row_codes) > 1 else 0.0,
                'top_probs': [
                    (str(row_codes[int(j)]), float(p))
                    for p, j in zip(top_vals.tolist(), top_idx.tolist())
                ],
            })
            print('\n[worst supervised target]')
            print({
                'target_candidate_index': int(target_idx),
                'target_action_code': str(row_codes[target_idx]),
                'target_display_line': str(feature.get('candidate_display_lines_in_order', [])[target_idx]) if feature.get('candidate_display_lines_in_order') else '',
            })
            print('\n[worst ordinalized_state_text]')
            print(str(feature.get('ordinalized_state_text', '')))
            print('\n[worst candidate_display_lines_in_order]')
            for idx, line in enumerate(feature.get('candidate_display_lines_in_order', [])):
                print({'candidate_idx': int(idx), 'line': str(line)})
            print('\n[worst candidate_query_texts]')
            for idx, text in enumerate(feature.get('candidate_query_texts', [])):
                print(f'--- query {int(idx)} / action_code={str(row_codes[idx])} ---')
                print(str(text))
                print()
    finally:
        if was_training:
            model.train()


class CandidateProbabilityProbeCallback(TrainerCallback):
    def __init__(self, probe_batch, probe_steps=200, max_rows=2, topk=5):
        self.probe_batch = probe_batch
        self.probe_steps = max(1, int(probe_steps))
        self.max_rows = max(1, int(max_rows))
        self.topk = max(1, int(topk))

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if model is None or self.probe_batch is None:
            return control
        if int(state.global_step) <= 0 or (int(state.global_step) % int(self.probe_steps)) != 0:
            return control
        try:
            _print_candidate_scoring_probe(
                model=model,
                batch=self.probe_batch,
                max_rows=self.max_rows,
                topk=self.topk,
                prefix=f'CANDIDATE PROBE step={int(state.global_step)}',
            )
        except Exception as exc:
            print(f'[CANDIDATE PROBE step={int(state.global_step)}] skipped: {exc}')
        return control
if CFG['enable_wandb']:
    project_name = CFG['wandb_project'] or f"{task_type}_optimization"
    wandb.init(project=project_name, name=f"{task_type}_{CFG['model_type']}_candidate_reranker_r{CFG['lora_r']}", config=CFG)
else:
    os.environ['WANDB_MODE'] = 'disabled'

with open(output_dir / 'training_hyperparams_args.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for k, v in CFG.items():
        writer.writerow([k, v])

candidate_model = CandidateScoringModel(
    backbone_model=model,
    score_token=str(CFG.get('candidate_score_token', '<CAND_SCORE>')),
    head_init_std=float(CFG.get('candidate_score_head_init_std', 0.02)),
    query_forward_batch_size=int(CFG.get('candidate_scoring_query_forward_batch_size', 16)),
)

trainable_params = sum(p.numel() for p in candidate_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in candidate_model.parameters())
print('candidate scoring model params:', {
    'trainable': int(trainable_params),
    'total': int(total_params),
})

data_collator = CandidateScoringCollator(tokenizer=tokenizer)

effective_bf16 = CFG['bf16'] if CFG['bf16'] is not None else (True if CFG['dtype'] == 'bfloat16' else False)
effective_fp16 = CFG['fp16'] if CFG['fp16'] is not None else (True if CFG['dtype'] == 'float16' else False)

training_args_kwargs = {
    'per_device_train_batch_size': CFG['per_device_train_batch_size'],
    'gradient_accumulation_steps': CFG['gradient_accumulation_steps'],
    'warmup_steps': CFG['warmup_steps'],
    'num_train_epochs': CFG['num_train_epochs'],
    'learning_rate': CFG['learning_rate'],
    'bf16': effective_bf16,
    'fp16': effective_fp16,
    'logging_steps': CFG['logging_steps'],
    'logging_first_step': True,
    'logging_nan_inf_filter': bool(CFG.get('logging_nan_inf_filter', False)),
    'optim': CFG['optim'],
    'weight_decay': CFG['weight_decay'],
    'lr_scheduler_type': CFG['lr_scheduler_type'],
    'seed': CFG['seed'],
    'output_dir': str(output_dir),
    'report_to': (CFG['report_to'] if CFG['enable_wandb'] else 'none'),
    'load_best_model_at_end': (CFG['load_best_model_at_end'] if enable_eval else False),
    'metric_for_best_model': CFG['metric_for_best_model'],
    'greater_is_better': CFG['greater_is_better'],
    'save_total_limit': CFG['save_total_limit'],
    'save_steps': CFG['save_steps'],
    'save_strategy': CFG['save_strategy'],
    'eval_strategy': eval_strategy_value,
    'eval_steps': eval_steps_value,
    'per_device_eval_batch_size': CFG['per_device_eval_batch_size'],
    'group_by_length': CFG['group_by_length'],
    'dataloader_num_workers': CFG['dataloader_num_workers'],
    'remove_unused_columns': CFG['remove_unused_columns'],
    'run_name': CFG['run_name'],
}


def _build_training_args_kwargs(training_args_cls, raw_kwargs):
    cleaned = {k: v for k, v in raw_kwargs.items() if v is not None}
    candidates = [dict(cleaned)]
    if 'eval_strategy' in cleaned and 'evaluation_strategy' not in cleaned:
        alt = dict(cleaned)
        alt['evaluation_strategy'] = alt.pop('eval_strategy')
        candidates.append(alt)
    if 'evaluation_strategy' in cleaned and 'eval_strategy' not in cleaned:
        alt = dict(cleaned)
        alt['eval_strategy'] = alt.pop('evaluation_strategy')
        candidates.append(alt)

    for candidate in candidates:
        try:
            training_args_obj = TrainingArguments(**candidate)
            return training_args_obj, candidate, []
        except TypeError:
            pass

    supported_training_arg_names = {
        name for name in inspect.signature(TrainingArguments.__init__).parameters.keys()
        if name != 'self'
    }
    fallback = {k: v for k, v in cleaned.items() if k in supported_training_arg_names}
    dropped = sorted(k for k in cleaned.keys() if k not in supported_training_arg_names)
    training_args_obj = TrainingArguments(**fallback)
    return training_args_obj, fallback, dropped

training_args_obj, final_training_args, dropped_training_args = _build_training_args_kwargs(TrainingArguments, training_args_kwargs)
if dropped_training_args:
    print('TrainingArgs unsupported keys dropped:', dropped_training_args)
print('resolved training args subset:', {
    'per_device_train_batch_size': final_training_args.get('per_device_train_batch_size'),
    'gradient_accumulation_steps': final_training_args.get('gradient_accumulation_steps'),
    'per_device_eval_batch_size': final_training_args.get('per_device_eval_batch_size'),
    'num_train_epochs': final_training_args.get('num_train_epochs'),
    'learning_rate': final_training_args.get('learning_rate'),
    'save_steps': final_training_args.get('save_steps'),
})

callbacks = []
if CFG.get('upload_on_save', False):
    callbacks.append(
        UploadCheckpointToHFCallback(
            repo_id=CFG['hf_model_repo_out'],
            token=HF_TOKEN,
            output_dir=output_dir,
            upload_source=CFG.get('upload_source', 'latest_checkpoint'),
        )
    )
    print(f"checkpoint auto-upload enabled -> {CFG['hf_model_repo_out']}")
else:
    print('checkpoint auto-upload disabled')

trainer = CandidateScoringTrainer(
    model=candidate_model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    args=training_args_obj,
    callbacks=callbacks if callbacks else None,
)
print('candidate reranker trainer ready')
print('DEBUG: trainer args logging_nan_inf_filter =', getattr(trainer.args, 'logging_nan_inf_filter', None))
print('DEBUG: trainer args group_by_length =', getattr(trainer.args, 'group_by_length', None))
print('DEBUG: trainer args batch_size =', getattr(trainer.args, 'per_device_train_batch_size', None))

debug_features = [train_dataset[_i] for _i in range(min(8, len(train_dataset)))]
if debug_features:
    debug_batch = data_collator(debug_features)
    tensor_shapes = {k: tuple(v.shape) for k, v in debug_batch.items() if torch.is_tensor(v)}
    print('DEBUG: collated debug batch shapes', tensor_shapes)
    print('DEBUG: debug batch candidate counts', debug_batch['candidate_counts'].tolist())
    _print_candidate_scoring_probe(
        model=candidate_model,
        batch=debug_batch,
        max_rows=min(8, len(debug_features)),
        topk=int(CFG.get('action_prob_probe_topk', 5)),
        prefix='DEBUG manual candidate scoring probe',
    )
    _print_worst_candidate_scoring_example(
        model=candidate_model,
        features=debug_features,
        batch=debug_batch,
        topk=int(CFG.get('action_prob_probe_topk', 5)),
    )

if bool(CFG.get('log_action_prob_during_train', False)) and debug_features:
    trainer.add_callback(
        CandidateProbabilityProbeCallback(
            probe_batch=debug_batch,
            probe_steps=int(CFG.get('action_prob_probe_steps', 200)),
            max_rows=int(CFG.get('action_prob_probe_rows', 2)),
            topk=int(CFG.get('action_prob_probe_topk', 5)),
        )
    )
    print('training-time candidate probability probe enabled:', {
        'steps': int(CFG.get('action_prob_probe_steps', 200)),
        'rows': int(CFG.get('action_prob_probe_rows', 2)),
        'topk': int(CFG.get('action_prob_probe_topk', 5)),
    })
else:
    print('training-time candidate probability probe disabled')

if CFG.get('resume_from_checkpoint'):
    print('resume_from_checkpoint is not implemented for [fix] candidate_scoring yet; training from scratch')
else:
    print('candidate_scoring trains from scratch in [fix] notebook')

print('DEBUG: entering trainer.train()', flush=True)
trainer.train()

final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

print('training done')
print('output_dir:', output_dir)
print('final_dir:', final_dir)
print('checkpoints:')
for p in sorted(output_dir.glob('checkpoint-*')):
    print(' -', p)


#### prob 체크

In [ ]:
import gc
import math
from pathlib import Path

import torch
from peft import PeftModel
from unsloth import FastLanguageModel

# cell 10에서 올라간 임시 학습 모델이 남아 있으면 먼저 정리
try:
    del model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


class ProbeCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(feature['input_ids']) for feature in features)
        max_action_count = max(len(feature.get('feasible_action_ids', [])) for feature in features)
        batch = {
            'input_ids': [],
            'attention_mask': [],
            'labels': [],
            'loss_weights': [],
            'action_target_mask': [],
            'feasible_action_ids': [],
        }
        for feature in features:
            seq_len = len(feature['input_ids'])
            pad_len = max_len - seq_len
            batch['input_ids'].append(list(feature['input_ids']) + [int(self.pad_token_id)] * pad_len)
            batch['attention_mask'].append(list(feature['attention_mask']) + [0] * pad_len)
            batch['labels'].append(list(feature['labels']) + [-100] * pad_len)
            batch['loss_weights'].append(list(feature['loss_weights']) + [0.0] * pad_len)
            batch['action_target_mask'].append(list(feature.get('action_target_mask', [])) + [0] * pad_len)
            feasible_action_ids = list(feature.get('feasible_action_ids', []))
            batch['feasible_action_ids'].append(feasible_action_ids + [-1] * (max_action_count - len(feasible_action_ids)))
        return {
            'input_ids': torch.tensor(batch['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(batch['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(batch['labels'], dtype=torch.long),
            'loss_weights': torch.tensor(batch['loss_weights'], dtype=torch.float32),
            'action_target_mask': torch.tensor(batch['action_target_mask'], dtype=torch.long),
            'feasible_action_ids': torch.tensor(batch['feasible_action_ids'], dtype=torch.long),
        }


# 실제 checkpoint 경로로 바꿀 것
ckpt_path = "/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_policy_dispatch_llama8b_r128_lmhead_embed_20260331_083243/checkpoint-11000"

if not Path(ckpt_path).exists():
    raise FileNotFoundError(f"checkpoint not found: {ckpt_path}")

base_model_name = MODEL_MAP[CFG['model_type']]
dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16


def _load_unsloth_model_with_chat_template_fallback(**load_kwargs):
    try:
        return FastLanguageModel.from_pretrained(**load_kwargs)
    except Exception as exc:
        if "additional_chat_templates" not in str(exc):
            raise
        retry_kwargs = dict(load_kwargs)
        retry_kwargs["local_files_only"] = True
        print("[Warn] additional_chat_templates 404; retrying with local_files_only=True using cached files.")
        return FastLanguageModel.from_pretrained(**retry_kwargs)


probe_model, probe_tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)

token_install = ensure_action_special_tokens(
    tokenizer=probe_tokenizer,
    model=probe_model,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
validate_action_tokenizer_installation(
    tokenizer=probe_tokenizer,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
print('token_install:', token_install)

probe_model = PeftModel.from_pretrained(probe_model, ckpt_path, is_trainable=False)
probe_model.eval()

probe_rows = [train_dataset[i] for i in range(min(8, len(train_dataset)))]
probe_collator = ProbeCollator(probe_tokenizer)
probe_batch = probe_collator(probe_rows)

device = next(probe_model.parameters()).device
probe_batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in probe_batch.items()}

model_inputs = {
    k: v for k, v in probe_batch.items()
    if k not in {'labels', 'loss_weights', 'action_target_mask', 'feasible_action_ids'}
}

with torch.no_grad():
    outputs = probe_model(**model_inputs)
    logits = outputs.get('logits') if isinstance(outputs, dict) else outputs.logits

shift_logits = logits[..., :-1, :].contiguous()
shift_labels = probe_batch['labels'][..., 1:].contiguous()
shift_action_mask = probe_batch['action_target_mask'][..., 1:].contiguous().bool()
feasible_action_ids = probe_batch['feasible_action_ids']

action_rows, action_cols = torch.nonzero(shift_action_mask, as_tuple=True)

print('[checkpoint-10000 action probe]')
for row_idx, col_idx in zip(action_rows.tolist(), action_cols.tolist()):
    row_feasible_ids = feasible_action_ids[row_idx]
    row_feasible_ids = row_feasible_ids[row_feasible_ids.ge(0)]
    feasible_ids = [int(x) for x in row_feasible_ids.tolist()]

    target_id = int(shift_labels[row_idx, col_idx].item())
    candidate_logits = shift_logits[row_idx, col_idx].index_select(
        0, row_feasible_ids.to(shift_logits.device, dtype=torch.long)
    )
    probs = torch.softmax(candidate_logits.float(), dim=-1)

    topk = min(5, probs.numel())
    top_probs, top_idx = torch.topk(probs, k=topk)
    top_tokens = [
        probe_tokenizer.decode([feasible_ids[int(i)]], skip_special_tokens=False)
        for i in top_idx.tolist()
    ]

    print({
        'row_idx': row_idx,
        'feasible_count': len(feasible_ids),
        'target_token': probe_tokenizer.decode([target_id], skip_special_tokens=False),
        'logit_std': float(candidate_logits.float().std().cpu().item()),
        'prob_max': float(probs.max().cpu().item()),
        'prob_min': float(probs.min().cpu().item()),
        'entropy': float((-(probs * probs.clamp_min(1e-12).log())).sum().cpu().item()),
        'uniform_entropy': float(math.log(len(feasible_ids))),
        'top_probs': list(zip(top_tokens, [float(x) for x in top_probs.tolist()])),
    })

In [ ]:
import torch

action_tokens = [
    "<A0001>", "<A0002>", "<A0003>", "<A0100>", "<A1000>", "<A9999>"
]
action_ids = [probe_tokenizer.convert_tokens_to_ids(tok) for tok in action_tokens]
print("action_ids:", list(zip(action_tokens, action_ids)))

inp = probe_model.get_input_embeddings().weight.detach().float().cpu()
out = probe_model.get_output_embeddings().weight.detach().float().cpu()

print("\n[input embedding pairwise L2]")
for i in range(len(action_ids)):
    for j in range(i + 1, len(action_ids)):
        d = torch.norm(inp[action_ids[i]] - inp[action_ids[j]], p=2).item()
        print(action_tokens[i], action_tokens[j], d)

print("\n[lm_head pairwise L2]")
for i in range(len(action_ids)):
    for j in range(i + 1, len(action_ids)):
        d = torch.norm(out[action_ids[i]] - out[action_ids[j]], p=2).item()
        print(action_tokens[i], action_tokens[j], d)

print("\n[input norms]")
for tok, tid in zip(action_tokens, action_ids):
    print(tok, torch.norm(inp[tid], p=2).item())

print("\n[lm_head norms]")
for tok, tid in zip(action_tokens, action_ids):
    print(tok, torch.norm(out[tid], p=2).item())

## 4) (선택) 학습 완료 모델 HF 업로드


#### 모델 drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

src_output_dir = Path(output_dir)  # colab_02에서 이미 정의됨
dst_root = Path('/content/drive/MyDrive/LLM_JSSP_checkpoints') / src_output_dir.name
dst_root.mkdir(parents=True, exist_ok=True)

checkpoints = sorted(
    [p for p in src_output_dir.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1])
)

if not checkpoints:
    print('No checkpoint folder found in output_dir:', src_output_dir)
else:
    latest_ckpt = checkpoints[-1]
    dst_ckpt = dst_root / latest_ckpt.name

    if dst_ckpt.exists():
        shutil.rmtree(dst_ckpt)
    shutil.copytree(latest_ckpt, dst_ckpt)

    print('copied latest checkpoint to drive:')
    print('src =', latest_ckpt)
    print('dst =', dst_ckpt)
# CFG['resume_from_checkpoint'] = '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_policy_dispatch_llama8b_r64/checkpoint-9000'

#### 데이터 drive

In [ ]:
from pathlib import Path
import shutil
import json

if 'step_dataset_path' not in globals() or not step_dataset_path:
    raise RuntimeError('step_dataset_path is not defined. Run the dataset/preprocessing cell first.')

src_dataset = Path(step_dataset_path)
if not src_dataset.exists():
    raise FileNotFoundError(f'dataset file not found: {src_dataset}')

backup_root = Path('/content/drive/MyDrive/LLM_JSSP_datasets')
if 'output_dir' in globals() and output_dir is not None:
    backup_root = backup_root / Path(output_dir).name
backup_root.mkdir(parents=True, exist_ok=True)

dst_dataset = backup_root / src_dataset.name
shutil.copy2(src_dataset, dst_dataset)

manifest = {
    'src_dataset': str(src_dataset),
    'dst_dataset': str(dst_dataset),
    'dataset_size_bytes': src_dataset.stat().st_size,
    'adapter_role': CFG.get('adapter_role'),
    'env_mode': CFG.get('env_mode'),
    'step_supervision_mode': CFG.get('step_supervision_mode'),
    'dataset_source': CFG.get('dataset_source'),
    'step_dataset_hf_repo': CFG.get('step_dataset_hf_repo'),
    'step_dataset_hf_file': CFG.get('step_dataset_hf_file'),
    'policy_step_dataset_hf_repo_dispatch': CFG.get('policy_step_dataset_hf_repo_dispatch'),
    'policy_step_dataset_hf_file_dispatch': CFG.get('policy_step_dataset_hf_file_dispatch'),
    'reason_step_dataset_hf_repo_dispatch': CFG.get('reason_step_dataset_hf_repo_dispatch'),
    'reason_step_dataset_hf_file_dispatch': CFG.get('reason_step_dataset_hf_file_dispatch'),
    'mixed_step_dataset_hf_repo_dispatch': CFG.get('mixed_step_dataset_hf_repo_dispatch'),
    'mixed_step_dataset_hf_file_dispatch': CFG.get('mixed_step_dataset_hf_file_dispatch'),
}
manifest_path = backup_root / 'dataset_backup_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('dataset copied to drive:')
print('src =', src_dataset)
print('dst =', dst_dataset)
print('manifest =', manifest_path)


#### HF에 모델 업로드

In [ ]:
if CFG['upload_after_train']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['hf_model_repo_out'], repo_type='model', private=False, exist_ok=True)

    output_dir = Path(resolve_output_dir(CFG, resolve_task_type(CFG)))
    upload_dir = resolve_upload_dir(output_dir, CFG['upload_source'], CFG['checkpoint_tag'])
    print('upload_dir:', upload_dir)

    upload_folder(
        repo_id=CFG['hf_model_repo_out'],
        repo_type='model',
        folder_path=str(upload_dir),
        token=HF_TOKEN,
        commit_message=f"Upload LoRA ({upload_dir.name}) from colab_02_full",
    )

    files = api.list_repo_files(repo_id=CFG['hf_model_repo_out'], repo_type='model')
    print(f"upload done: {CFG['hf_model_repo_out']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_after_train]=False, skip upload')
